# 🔲 Corner Detection: A Comprehensive Tutorial

---

**Topics covered:**
- What is a corner and why does it matter?
- Intuition behind corner detection
- The Harris Corner Detector (mathematical derivation)
- The Shi-Tomasi (Good Features to Track) method
- FAST corner detector
- Comparing detectors
- **Applications in Medical Imaging**
- Summary & further reading

> 💡 **Prerequisites:** Basic knowledge of image gradients, NumPy, and OpenCV.

## 1. What is a Corner?

In image processing, a **corner** is a point where the image intensity changes significantly in **multiple directions**. This distinguishes corners from:

| Feature Type | Description |
|---|---|
| **Flat region** | No intensity change in any direction |
| **Edge** | Intensity changes only along one direction |
| **Corner** | Intensity changes in **two or more** directions |

Corners are extremely useful because they are:
- **Distinctive** — easy to localize precisely
- **Stable** — robust to small rotations and illumination changes
- **Sparse** — only a few corners per image, making them efficient

They are the foundation of many higher-level computer vision tasks: feature matching, object tracking, SLAM, panorama stitching, and medical image registration.

## 2. Intuition: The Sliding Window

The core idea behind corner detection is the **sliding window test**:

> *Slide a small window over the image. If shifting the window in **any direction** causes a large change in pixel intensities inside the window — you are at a corner.*

Mathematically, we define the **change in appearance** when a window $W$ is shifted by $(u, v)$:

$$E(u,v) = \sum_{(x,y) \in W} \left[ I(x+u, y+v) - I(x,y) \right]^2$$

Where:
- $I(x, y)$ is the image intensity at pixel $(x, y)$
- $(u, v)$ is the window shift
- $W$ is the set of pixels inside the window

Using **Taylor expansion** for small shifts, we approximate:
$$I(x+u, y+v) \approx I(x,y) + I_x \cdot u + I_y \cdot v$$

Substituting:
$$E(u,v) \approx \sum_{(x,y) \in W} \left[ I_x u + I_y v \right]^2$$

This can be written in matrix form:
$$E(u,v) \approx \begin{bmatrix} u & v \end{bmatrix} M \begin{bmatrix} u \\ v \end{bmatrix}$$

Where **M** is the **Structure Tensor** (or second-moment matrix):
$$M = \sum_{(x,y) \in W} \begin{bmatrix} I_x^2 & I_x I_y \\ I_x I_y & I_y^2 \end{bmatrix}$$

## 3. Environment Setup

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.colors import Normalize
from scipy.ndimage import gaussian_filter
from skimage import data, img_as_float, img_as_ubyte
import skimage.color # Explicitly import skimage.color
from skimage.feature import corner_harris, corner_peaks, corner_shi_tomasi, corner_foerstner
from skimage.draw import disk

# Consistent style
plt.rcParams['figure.dpi'] = 120
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.labelsize'] = 11
plt.rcParams['font.family'] = 'DejaVu Sans'

print("✅ All libraries loaded successfully!")
print(f"  OpenCV  : {cv2.__version__}")
print(f"  NumPy   : {np.__version__}")

## 4. Visualizing Flat, Edge, and Corner Regions

Before diving into detectors, let's build intuition by constructing a synthetic image with clearly flat regions, edges, and corners, and visualize how the gradient energy ($E$) behaves in each.

In [ ]:
# ─── Build a synthetic test image ───────────────────────────────────────────
img = np.zeros((200, 200), dtype=np.float32)
img[60:140, 60:140] = 1.0          # white square → 4 corners, 4 edges, 1 flat centre
img = gaussian_filter(img, sigma=1) # slight blur to make gradients nicer

# Compute image gradients
Ix = np.gradient(img, axis=1)
Iy = np.gradient(img, axis=0)

# Structure tensor components
Ixx = gaussian_filter(Ix**2,    sigma=2)
Iyy = gaussian_filter(Iy**2,    sigma=2)
Ixy = gaussian_filter(Ix * Iy,  sigma=2)

# Harris response (preview – full derivation next section)
k = 0.04
det_M  = Ixx * Iyy - Ixy**2
trace_M = Ixx + Iyy
R = det_M - k * trace_M**2

# ─── Plot ───────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

axes[0].imshow(img, cmap='gray')
axes[0].set_title('Synthetic Image')
axes[0].axis('off')

axes[1].imshow(np.sqrt(Ix**2 + Iy**2), cmap='hot')
axes[1].set_title('Gradient Magnitude')
axes[1].axis('off')

im = axes[2].imshow(R, cmap='RdYlGn')
axes[2].set_title('Harris Response R\n(green = corner, red = edge)')
axes[2].axis('off')
fig.colorbar(im, ax=axes[2], fraction=0.046)

# Annotate region types
overlay = np.stack([img]*3, axis=-1)
axes[3].imshow(img, cmap='gray')
for (y, x, label, color) in [
    (30,  30,  'Flat\n(background)', 'cyan'),
    (100, 60,  'Edge\n(left side)',   'yellow'),
    (60,  60,  'Corner\n(top-left)', 'red'),
]:
    axes[3].annotate(label, xy=(x, y), fontsize=8, color=color,
                     bbox=dict(boxstyle='round,pad=0.2', fc='black', alpha=0.5))
    axes[3].plot(x, y, 'o', color=color, markersize=8)
axes[3].set_title('Region Types')
axes[3].axis('off')

plt.suptitle('Understanding Corners vs. Edges vs. Flat Regions', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 5. The Harris Corner Detector

### 5.1 Mathematical Derivation

The Harris detector (Harris & Stephens, 1988) avoids computing eigenvalues explicitly by using a **corner response function** derived from the structure tensor $M$.

Recall $M = \begin{bmatrix} \sum I_x^2 & \sum I_x I_y \\ \sum I_x I_y & \sum I_y^2 \end{bmatrix}$

The eigenvalues $\lambda_1, \lambda_2$ of $M$ encode local structure:

| $\lambda_1$ | $\lambda_2$ | Interpretation |
|---|---|---|
| Small | Small | Flat region |
| Large | Small | Edge |
| Large | Large | **Corner** ✅ |

Instead of computing eigenvalues (expensive), Harris defines:

$$\boxed{R = \det(M) - k \cdot \text{trace}(M)^2}$$

Where:
- $\det(M) = \lambda_1 \lambda_2$
- $\text{trace}(M) = \lambda_1 + \lambda_2$
- $k$ is a sensitivity parameter (typically $0.04$–$0.06$)

**Decision rule:**
- $R \gg 0$ → **Corner**
- $R \approx 0$ → **Flat region**
- $R \ll 0$ → **Edge**

In [ ]:
# ─── Visualize the λ1-λ2 decision space ────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 6))

lam = np.linspace(0, 3, 400)
L1, L2 = np.meshgrid(lam, lam)
k = 0.04
R_space = (L1 * L2) - k * (L1 + L2)**2

cf = ax.contourf(L1, L2, R_space, levels=50, cmap='RdYlGn')
plt.colorbar(cf, ax=ax, label='Harris R value')
ax.contour(L1, L2, R_space, levels=[0], colors='black', linewidths=2)

ax.text(0.3, 0.3, 'FLAT\n(small λ₁, λ₂)', ha='center', fontsize=10, color='gray')
ax.text(2.2, 0.3, 'EDGE\n(large λ₁, small λ₂)', ha='center', fontsize=10, color='black')
ax.text(2.0, 2.2, 'CORNER\n(large λ₁, large λ₂)', ha='center', fontsize=10,
        color='darkgreen', fontweight='bold')

ax.set_xlabel('λ₁  (gradient energy in direction 1)')
ax.set_ylabel('λ₂  (gradient energy in direction 2)')
ax.set_title('Harris Decision Space\n(green = corner, red = edge, grey = flat)')
plt.tight_layout()
plt.show()

### 5.2 Implementing Harris from Scratch

In [ ]:
def harris_corner_response(image_gray, k=0.04, sigma=1.5):
    """
    Compute the Harris corner response map.

    Parameters
    ----------
    image_gray : 2D ndarray, float
    k          : Harris sensitivity constant (0.04 – 0.06)
    sigma      : Gaussian smoothing sigma for structure tensor

    Returns
    -------
    R : 2D ndarray — Harris response (same shape as input)
    """
    # Step 1: Compute image gradients (Sobel)
    Ix = cv2.Sobel(image_gray, cv2.CV_64F, 1, 0, ksize=3)
    Iy = cv2.Sobel(image_gray, cv2.CV_64F, 0, 1, ksize=3)

    # Step 2: Compute structure tensor elements, smoothed by Gaussian
    Ixx = gaussian_filter(Ix**2,    sigma=sigma)
    Iyy = gaussian_filter(Iy**2,    sigma=sigma)
    Ixy = gaussian_filter(Ix * Iy,  sigma=sigma)

    # Step 3: Harris response R = det(M) - k * trace(M)^2
    det_M   = Ixx * Iyy - Ixy**2
    trace_M = Ixx + Iyy
    R = det_M - k * (trace_M**2)

    return R


def detect_corners(R, threshold_rel=0.01, min_distance=10):
    """
    Extract corner coordinates from a Harris response map using
    Non-Maximum Suppression.

    Parameters
    ----------
    R             : Harris response map
    threshold_rel : fraction of max(R) to use as threshold
    min_distance  : minimum pixels between corners

    Returns
    -------
    corners : array of (row, col) coordinates
    """
    from skimage.feature import peak_local_max
    threshold_abs = threshold_rel * R.max()
    corners = peak_local_max(R,
                              min_distance=min_distance,
                              threshold_abs=threshold_abs)
    return corners


# ─── Test on a real image ────────────────────────────────────────────────────
image_color = data.chelsea()             # standard skimage test image
image_gray  = skimage.color.rgb2gray(image_color).astype(np.float32)

R = harris_corner_response(image_gray, k=0.04, sigma=2)
corners = detect_corners(R, threshold_rel=0.02, min_distance=8)

print(f"Detected {len(corners)} corners")

# ─── Plot ─────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].imshow(image_color)
axes[0].set_title('Original Image')
axes[0].axis('off')

axes[1].imshow(R, cmap='hot')
axes[1].set_title('Harris Response Map R')
axes[1].axis('off')

axes[2].imshow(image_color)
axes[2].plot(corners[:, 1], corners[:, 0], 'r+', markersize=6, markeredgewidth=1.5)
axes[2].set_title(f'Harris Corners Detected ({len(corners)})')
axes[2].axis('off')

plt.suptitle('Harris Corner Detector — From Scratch', fontsize=14)
plt.tight_layout()
plt.show()

### 5.3 Effect of the Sensitivity Parameter `k`

The parameter $k$ controls the trade-off between detecting corners and edges:
- **Low $k$** → more permissive, detects more corners (but more false positives)
- **High $k$** → more restrictive, fewer but more reliable corners

In [ ]:
k_values = [0.01, 0.04, 0.08, 0.15]
fig, axes = plt.subplots(1, len(k_values), figsize=(16, 4))

for ax, k_val in zip(axes, k_values):
    R_k = harris_corner_response(image_gray, k=k_val, sigma=2)
    corners_k = detect_corners(R_k, threshold_rel=0.02, min_distance=8)
    ax.imshow(image_color)
    ax.plot(corners_k[:, 1], corners_k[:, 0], 'r+', markersize=5, markeredgewidth=1.5)
    ax.set_title(f'k = {k_val}\n({len(corners_k)} corners)')
    ax.axis('off')

plt.suptitle('Effect of Sensitivity Parameter k on Harris Detection', fontsize=13)
plt.tight_layout()
plt.show()

## 6. Shi-Tomasi: "Good Features to Track"

Shi & Tomasi (1994) proposed a simpler, more robust corner criterion.
Instead of the Harris algebraic approximation, they directly use the **smaller eigenvalue**:

$$\boxed{R_{\text{ST}} = \min(\lambda_1, \lambda_2)}$$

A point is a corner if $\min(\lambda_1, \lambda_2) > \text{threshold}$.

**Why is this better?**
- More uniform distribution of detected corners
- Stronger theoretical guarantees for tracking stability
- The default detector in OpenCV's `goodFeaturesToTrack()`

In [ ]:
# ─── Shi-Tomasi via OpenCV ───────────────────────────────────────────────────
image_uint8 = img_as_ubyte(image_gray)

shi_corners = cv2.goodFeaturesToTrack(
    image_uint8,
    maxCorners=200,
    qualityLevel=0.01,    # min quality = 1% of best corner response
    minDistance=8,        # minimum Euclidean distance between corners
    blockSize=3
)
shi_corners = shi_corners.reshape(-1, 2)  # shape: (N, 2) with [x, y]

# ─── Shi-Tomasi via scikit-image (for response map) ─────────────────────────
R_st = corner_shi_tomasi(image_gray)

# ─── Plot comparison ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].imshow(R_st, cmap='hot')
axes[0].set_title('Shi-Tomasi Response\nmin(λ₁, λ₂)')
axes[0].axis('off')

axes[1].imshow(image_color)
axes[1].plot(shi_corners[:, 0], shi_corners[:, 1], 'g+',
             markersize=7, markeredgewidth=1.5)
axes[1].set_title(f'Shi-Tomasi Corners ({len(shi_corners)})')
axes[1].axis('off')

# Side-by-side with Harris
axes[2].imshow(image_color)
axes[2].plot(corners[:, 1], corners[:, 0], 'r+',
             markersize=7, markeredgewidth=1.5, label='Harris')
axes[2].plot(shi_corners[:, 0], shi_corners[:, 1], 'g^',
             markersize=5, markeredgewidth=1, label='Shi-Tomasi', alpha=0.7)
axes[2].legend(loc='upper right', fontsize=9)
axes[2].set_title('Harris (red) vs Shi-Tomasi (green)')
axes[2].axis('off')

plt.suptitle('Shi-Tomasi Corner Detector', fontsize=14)
plt.tight_layout()
plt.show()

## 7. FAST: Features from Accelerated Segment Test

FAST (Rosten & Drummond, 2006) is a **real-time corner detector** based on pixel intensity comparisons on a Bresenham circle, not gradients.

### Algorithm

For each candidate pixel $p$ with intensity $I_p$:
1. Examine 16 pixels on a circle of radius 3 around $p$ (Bresenham circle)
2. $p$ is a corner if **at least $N$ contiguous pixels** on the circle are all:
   - Brighter than $I_p + t$, OR
   - Darker than $I_p - t$
3. Typical choice: $N = 12$ (FAST-12)

**Speed trick:** Test positions 1, 5, 9, 13 first (compass points). If fewer than 3 are much brighter/darker → not a corner → skip immediately.

### Advantages
- **Extremely fast** — 10–50× faster than Harris
- Suitable for real-time video and mobile applications

### Disadvantages  
- Sensitive to noise
- Not rotation or scale invariant (unlike SIFT/SURF)
- No sub-pixel accuracy

In [ ]:
# ─── FAST via OpenCV ─────────────────────────────────────────────────────────
fast = cv2.FastFeatureDetector_create()

fast.setThreshold(20)          # intensity difference threshold
fast.setNonmaxSuppression(True)

keypoints_fast = fast.detect(image_uint8, None)

# Draw keypoints
img_fast = cv2.drawKeypoints(image_color,
                              keypoints_fast,
                              None,
                              color=(255, 0, 0),
                              flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)

# ─── Effect of threshold on FAST ────────────────────────────────────────────
thresholds = [5, 20, 50, 100]
fig, axes = plt.subplots(1, len(thresholds), figsize=(16, 4))

for ax, t in zip(axes, thresholds):
    fast.setThreshold(t)
    kps = fast.detect(image_uint8, None)
    pts = np.array([[kp.pt[0], kp.pt[1]] for kp in kps])
    ax.imshow(image_color)
    if len(pts):
        ax.plot(pts[:, 0], pts[:, 1], 'b.', markersize=3)
    ax.set_title(f'FAST threshold={t}\n({len(kps)} corners)')
    ax.axis('off')

plt.suptitle('FAST Detector — Effect of Intensity Threshold', fontsize=13)
plt.tight_layout()
plt.show()

### 7.1 Visualizing the FAST Circle Test

In [ ]:
# ─── Bresenham circle visualization ─────────────────────────────────────────
fig, ax = plt.subplots(figsize=(5, 5))

# Pixel grid
grid = np.ones((9, 9)) * 0.6
cx, cy = 4, 4

# Bresenham circle of radius 3 (16 pixels)
circle_offsets = [
    (0, 3),(1, 3),(2, 2),(3, 1),(3, 0),(3,-1),(2,-2),(1,-3),
    (0,-3),(-1,-3),(-2,-2),(-3,-1),(-3,0),(-3,1),(-2,2),(-1,3)
]

# Simulate: first 12 are darker (corner)
for i, (dx, dy) in enumerate(circle_offsets):
    color = 0.15 if i < 12 else 0.85  # darker patch = potential corner
    grid[cy + dy, cx + dx] = color

grid[cy, cx] = 0.5  # centre pixel p

ax.imshow(grid, cmap='gray', vmin=0, vmax=1)
ax.set_xticks(np.arange(-0.5, 9, 1), minor=True)
ax.set_yticks(np.arange(-0.5, 9, 1), minor=True)
ax.grid(which='minor', color='black', linewidth=0.8)
ax.tick_params(which='both', bottom=False, left=False,
               labelbottom=False, labelleft=False)

# Label compass points
for label, (dx, dy) in zip(['1','5','9','13'],
                             [(0,3),(3,0),(0,-3),(-3,0)]):
    ax.text(cx + dx, cy + dy, label, ha='center', va='center',
            fontsize=10, color='red', fontweight='bold')

ax.text(cx, cy, 'p', ha='center', va='center',
        fontsize=12, color='lime', fontweight='bold')

ax.set_title('FAST: Bresenham Circle (radius=3)\n'
             'Dark = pixels darker than p – t\n'
             'Red labels = fast-check compass points', fontsize=9)
plt.tight_layout()
plt.show()

## 8. Comparing All Three Detectors

Let's put Harris, Shi-Tomasi, and FAST side by side on a structured image and a natural image.

In [ ]:
def run_all_detectors(image_color_in):
    gray = skimage.color.rgb2gray(image_color_in).astype(np.float32)
    uint8 = img_as_ubyte(gray)

    # Harris
    R_h = harris_corner_response(gray, k=0.04, sigma=2)
    c_harris = detect_corners(R_h, threshold_rel=0.02, min_distance=8)

    # Shi-Tomasi
    c_st = cv2.goodFeaturesToTrack(uint8, maxCorners=300,
                                    qualityLevel=0.01, minDistance=8)
    c_st = c_st.reshape(-1, 2) if c_st is not None else np.empty((0,2))

    # FAST
    det = cv2.FastFeatureDetector_create(threshold=20, nonmaxSuppression=True)
    kps = det.detect(uint8, None)
    c_fast = np.array([[kp.pt[0], kp.pt[1]] for kp in kps]) if kps else np.empty((0,2))

    return c_harris, c_st, c_fast


test_images = [
    ('Checkerboard', data.checkerboard()),
    ('Camera',       data.camera()),
]

fig, axes = plt.subplots(len(test_images), 4, figsize=(16, 4 * len(test_images)))

for row, (name, img_in) in enumerate(test_images):
    if img_in.ndim == 2:
        img_color = np.stack([img_in]*3, axis=-1)
    else:
        img_color = img_in

    c_h, c_st, c_f = run_all_detectors(img_color)

    axes[row, 0].imshow(img_color, cmap='gray')
    axes[row, 0].set_title(f'{name}\nOriginal')
    axes[row, 0].axis('off')

    for col, (corners, label, marker, clr) in enumerate([
        (c_h,  f'Harris\n({len(c_h)} pts)',       '+', 'red'),
        (c_st, f'Shi-Tomasi\n({len(c_st)} pts)',  '^', 'lime'),
        (c_f,  f'FAST\n({len(c_f)} pts)',          '.', 'cyan'),
    ]):
        ax = axes[row, col + 1]
        ax.imshow(img_color, cmap='gray')
        if len(corners):
            if corners.shape[1] == 2 and col == 0:   # Harris: (row, col)
                ax.plot(corners[:, 1], corners[:, 0], marker,
                        color=clr, markersize=5, markeredgewidth=1.2)
            else:                                      # ST, FAST: (x, y)
                ax.plot(corners[:, 0], corners[:, 1], marker,
                        color=clr, markersize=5, markeredgewidth=1.2)
        ax.set_title(label)
        ax.axis('off')

plt.suptitle('Harris vs Shi-Tomasi vs FAST — Side-by-Side Comparison', fontsize=14)
plt.tight_layout()
plt.show()

### Detector Comparison Summary

| Property | Harris | Shi-Tomasi | FAST |
|---|---|---|---|
| **Response function** | det(M) – k·trace²(M) | min(λ₁, λ₂) | Intensity circle test |
| **Speed** | Medium | Medium | ⚡ Very fast |
| **Sub-pixel accuracy** | ✅ (with refinement) | ✅ | ❌ |
| **Rotation invariant** | ✅ | ✅ | ❌ |
| **Scale invariant** | ❌ | ❌ | ❌ |
| **Best use case** | General feature detection | Tracking, homography | Real-time / mobile |

---
## 🏥 9. Corner Detection in Medical Imaging

Corner detection plays a **critical role** in medical image analysis. Anatomical structures in medical images (e.g., vessels, organ boundaries, bone edges, lesions) form distinctive corners and junctions that can be exploited for:

| Application | How corners help |
|---|---|
| **Image Registration** | Align pre- and post-operative scans by matching corner features |
| **Retinal vessel analysis** | Detect bifurcation points (junctions of blood vessels) |
| **Bone/joint analysis** | Localize trabecular edges in X-ray/CT |
| **Cell detection in histology** | Nuclei and cell membrane corners mark cell boundaries |
| **Lesion boundary delineation** | Corners mark inflection points on tumor contours |
| **Endoscopy / laparoscopy tracking** | Real-time FAST corners for instrument tracking |

Let's demonstrate two concrete examples:

### 9.1 — Retinal Vessel Bifurcation Detection (simulated)

In [ ]:
# ─── Simulate a retinal vessel-like image ───────────────────────────────────
retina = np.zeros((300, 300), dtype=np.float32)

# Draw vessel tree
def draw_vessel(img, y0, x0, y1, x1, width=4):
    """Draw a vessel segment as a thick line."""
    rr, cc = np.linspace(y0, y1, 500).astype(int), np.linspace(x0, x1, 500).astype(int)
    for dy in range(-width, width+1):
        for dx in range(-width, width+1):
            yr = np.clip(rr + dy, 0, img.shape[0]-1)
            xc = np.clip(cc + dx, 0, img.shape[1]-1)
            img[yr, xc] = 0.85

# Main vessel (trunk)
draw_vessel(retina, 150, 20,  150, 280, width=6)
# Branch 1 (upward)
draw_vessel(retina, 150, 120,  60, 240, width=4)
# Branch 2 (downward)
draw_vessel(retina, 150, 120, 240, 200, width=4)
# Sub-branch
draw_vessel(retina, 150, 200,  80, 280, width=2)
draw_vessel(retina, 150, 200, 220, 270, width=2)

# Add realistic noise
retina += np.random.normal(0, 0.05, retina.shape).astype(np.float32)
retina = np.clip(retina, 0, 1)
retina_blurred = gaussian_filter(retina, sigma=1.5)

# Apply Harris detection
R_retina = harris_corner_response(retina_blurred, k=0.04, sigma=2)
corners_retina = detect_corners(R_retina, threshold_rel=0.05, min_distance=15)

# ─── Plot ─────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(retina_blurred, cmap='gray')
axes[0].set_title('Simulated Retinal Vessel Map')
axes[0].axis('off')

axes[1].imshow(R_retina, cmap='hot')
axes[1].set_title('Harris Corner Response')
axes[1].axis('off')

axes[2].imshow(retina_blurred, cmap='gray')
axes[2].plot(corners_retina[:, 1], corners_retina[:, 0],
             'r*', markersize=14, label='Bifurcation points')
axes[2].set_title(f'Detected Bifurcation Points ({len(corners_retina)})')
axes[2].legend(fontsize=9)
axes[2].axis('off')

plt.suptitle('🏥 Medical Imaging Application: Retinal Vessel Bifurcation Detection',
             fontsize=13)
plt.tight_layout()
plt.show()

print("Clinical relevance: Bifurcation points (junctions where vessels split)")
print("are anatomical landmarks used for retinal image registration,")
print("glaucoma progression monitoring, and diabetic retinopathy staging.")

### 9.2 — Cell Nucleus Corner Detection in Histology

In [ ]:
from skimage.draw import ellipse

# ─── Simulate a histology-like image with cell nuclei ───────────────────────
np.random.seed(42)
histo = np.ones((300, 300), dtype=np.float32) * 0.85  # pale pink background

nuclei_centers = []
for _ in range(25):
    cy_n = np.random.randint(20, 280)
    cx_n = np.random.randint(20, 280)
    r_major = np.random.randint(8, 16)
    r_minor = np.random.randint(6, 12)
    angle   = np.random.uniform(0, np.pi)
    rr, cc  = ellipse(cy_n, cx_n, r_major, r_minor,
                       shape=histo.shape, rotation=angle)
    histo[rr, cc] = np.random.uniform(0.15, 0.35)  # dark purple nuclei
    nuclei_centers.append((cy_n, cx_n))

# Slight blur + noise
histo += np.random.normal(0, 0.03, histo.shape).astype(np.float32)
histo = np.clip(histo, 0, 1)
histo_blur = gaussian_filter(histo, sigma=1.0)

# Detect corners (will cluster around nucleus boundaries)
R_histo = harris_corner_response(histo_blur, k=0.04, sigma=1.5)
corners_histo = detect_corners(R_histo, threshold_rel=0.01, min_distance=6)

# ─── Plot ─────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(histo_blur, cmap='gray')
axes[0].set_title('Simulated H&E Histology Image\n(dark ellipses = cell nuclei)')
axes[0].axis('off')

axes[1].imshow(R_histo, cmap='hot')
axes[1].set_title('Harris Response at Cell Boundaries')
axes[1].axis('off')

axes[2].imshow(histo_blur, cmap='gray')
axes[2].plot(corners_histo[:, 1], corners_histo[:, 0],
             'y+', markersize=8, markeredgewidth=1.5, label='Boundary corners')
axes[2].plot([c[1] for c in nuclei_centers], [c[0] for c in nuclei_centers],
             'ro', markersize=5, alpha=0.5, label='True nucleus centres')
axes[2].legend(fontsize=8)
axes[2].set_title(f'Corner-Based Cell Boundary Detection\n({len(corners_histo)} corners found)')
axes[2].axis('off')

plt.suptitle('🏥 Medical Imaging Application: Histology Cell Nucleus Boundary Detection',
             fontsize=13)
plt.tight_layout()
plt.show()

print("Clinical relevance: Automated cell counting and nucleus boundary detection")
print("supports cancer grading, mitosis detection, and spatial cell distribution")
print("analysis in digital pathology pipelines.")

### 9.3 — Image Registration using Corner Matching (MRI-style)

In [ ]:
from skimage.transform import warp, AffineTransform

# ─── Simulate two MRI slices: reference and slightly shifted ─────────────────
mri_ref = data.brain()[..., 0].astype(np.float32)  # skimage built-in MRI slice
mri_ref = mri_ref / mri_ref.max()

# Apply a small affine transform to simulate patient motion
tform = AffineTransform(translation=(8, 5), rotation=0.04)
mri_moved = warp(mri_ref, tform.inverse, mode='edge')

# Detect Shi-Tomasi corners in both
def shi_tomasi_corners(gray_img, n=150, quality=0.01, dist=8):
    u8 = img_as_ubyte(np.clip(gray_img, 0, 1))
    pts = cv2.goodFeaturesToTrack(u8, maxCorners=n,
                                   qualityLevel=quality, minDistance=dist)
    return pts.reshape(-1, 2) if pts is not None else np.empty((0,2))

c_ref   = shi_tomasi_corners(mri_ref)
c_moved = shi_tomasi_corners(mri_moved)

# ─── Plot ─────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(mri_ref, cmap='gray')
axes[0].plot(c_ref[:, 0], c_ref[:, 1], 'g+', markersize=6)
axes[0].set_title(f'Reference MRI Slice\n({len(c_ref)} Shi-Tomasi corners)')
axes[0].axis('off')

axes[1].imshow(mri_moved, cmap='gray')
axes[1].plot(c_moved[:, 0], c_moved[:, 1], 'r+', markersize=6)
axes[1].set_title(f'Moved MRI Slice (shift+rotate)\n({len(c_moved)} corners)')
axes[1].axis('off')

# Show overlay difference
diff = np.abs(mri_ref - mri_moved)
axes[2].imshow(diff, cmap='hot')
axes[2].set_title('Difference Map\n(corners would drive alignment correction)')
axes[2].axis('off')

plt.suptitle('🏥 Medical Imaging Application: Corner-based MRI Registration',
             fontsize=13)
plt.tight_layout()
plt.show()

print("Clinical relevance: Corner feature matching is used in affine/non-rigid")
print("registration of multi-modal scans (MRI-CT, pre/post-treatment), enabling")
print("longitudinal disease progression analysis and surgical planning.")

## 10. Sub-Pixel Corner Refinement

Detected corners are initially at integer pixel locations. For precision applications (medical imaging, metrology, camera calibration), we need **sub-pixel accuracy**.

OpenCV's `cornerSubPix` iteratively shifts each corner to the position where the gradient dot products with the displacement vectors are zero:

$$\sum_i \left[ \nabla I(q_i) \cdot (q_i - q) \right] = 0$$

In [ ]:
# ─── Sub-pixel refinement on checkerboard ────────────────────────────────────
checker = img_as_ubyte(data.checkerboard())

# Initial integer-precision Shi-Tomasi
corners_int = cv2.goodFeaturesToTrack(checker, maxCorners=50,
                                       qualityLevel=0.05, minDistance=15)

# Refine to sub-pixel
criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 40, 0.001)
corners_sub = cv2.cornerSubPix(checker, corners_int.copy(),
                                 winSize=(5, 5), zeroZone=(-1, -1),
                                 criteria=criteria)

# ─── Compare pixel vs sub-pixel ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, (corners, title) in zip(axes, [
    (corners_int, 'Integer-Precision Corners'),
    (corners_sub, 'Sub-Pixel Refined Corners'),
]):
    ax.imshow(checker, cmap='gray')
    pts = corners.reshape(-1, 2)
    ax.plot(pts[:, 0], pts[:, 1], 'r+', markersize=12, markeredgewidth=2)
    # Zoom-in inset
    ax_ins = ax.inset_axes([0.6, 0.6, 0.38, 0.38])
    ax_ins.imshow(checker[50:110, 50:110], cmap='gray')
    local = pts[(pts[:, 0] > 50) & (pts[:, 0] < 110) &
                (pts[:, 1] > 50) & (pts[:, 1] < 110)]
    ax_ins.plot(local[:, 0] - 50, local[:, 1] - 50,
                'r+', markersize=12, markeredgewidth=2)
    ax_ins.set_xticks([]); ax_ins.set_yticks([])
    ax.set_title(title)
    ax.axis('off')

# Print displacement stats
displacements = np.linalg.norm(corners_sub - corners_int, axis=2)
print(f"Sub-pixel refinement statistics:")
print(f"  Mean displacement : {displacements.mean():.4f} pixels")
print(f"  Max  displacement : {displacements.max():.4f} pixels")

plt.suptitle('Sub-Pixel Corner Refinement using OpenCV cornerSubPix', fontsize=13)
plt.tight_layout()
plt.show()

## 11. Robustness to Noise and Blur

Real images (especially medical ones) suffer from noise and blur. Let's evaluate how detectors degrade.

In [ ]:
base = skimage.color.rgb2gray(data.chelsea()).astype(np.float32)

noise_levels = [0.0, 0.02, 0.05, 0.10]
fig, axes = plt.subplots(2, len(noise_levels), figsize=(16, 7))

for col, sigma_n in enumerate(noise_levels):
    noisy = np.clip(base + np.random.normal(0, sigma_n, base.shape).astype(np.float32),
                    0, 1)
    R_noisy = harris_corner_response(noisy, k=0.04, sigma=2)
    corners_n = detect_corners(R_noisy, threshold_rel=0.02, min_distance=8)

    axes[0, col].imshow(noisy, cmap='gray')
    axes[0, col].set_title(f'σ_noise = {sigma_n}')
    axes[0, col].axis('off')

    axes[1, col].imshow(np.stack([noisy]*3, axis=-1))
    if len(corners_n):
        axes[1, col].plot(corners_n[:, 1], corners_n[:, 0],
                          'r+', markersize=5, markeredgewidth=1.2)
    axes[1, col].set_title(f'{len(corners_n)} corners')
    axes[1, col].axis('off')

plt.suptitle('Harris Corner Detection Robustness to Additive Noise', fontsize=13)
plt.tight_layout()
plt.show()

## 12. Summary

### Key Takeaways

```
┌─────────────────────────────────────────────────────────────────┐
│                    Corner Detection Summary                     │
├─────────────────┬───────────────────────────────────────────────┤
│ Core Idea       │ Regions with large intensity change in ≥ 2   │
│                 │ directions (large eigenvalues of M)           │
├─────────────────┼───────────────────────────────────────────────┤
│ Harris          │ R = det(M) – k·trace(M)²                      │
│                 │ Fast, classic, slightly biased on edges        │
├─────────────────┼───────────────────────────────────────────────┤
│ Shi-Tomasi      │ R = min(λ₁, λ₂)                              │
│                 │ More uniform, better for tracking              │
├─────────────────┼───────────────────────────────────────────────┤
│ FAST            │ Bresenham circle intensity test               │
│                 │ 10–50× faster, real-time applications         │
├─────────────────┼───────────────────────────────────────────────┤
│ Medical Use     │ Vessel bifurcations, cell boundaries,         │
│                 │ image registration, lesion delineation         │
└─────────────────┴───────────────────────────────────────────────┘
```

### Further Reading

1. **Harris & Stephens (1988)** — *A Combined Corner and Edge Detector* — the original paper
2. **Shi & Tomasi (1994)** — *Good Features to Track* — CVPR
3. **Rosten & Drummond (2006)** — *Machine learning for high-speed corner detection* — ECCV
4. **Szeliski (2022)** — *Computer Vision: Algorithms and Applications* — Ch. 7 (freely available)
5. **OpenCV Documentation** — `cv2.cornerHarris`, `cv2.goodFeaturesToTrack`, `cv2.FastFeatureDetector`

### What's Next?

- **Scale-invariant features:** SIFT, SURF, ORB (combine corner detection with descriptors)
- **Feature matching:** FLANN, BFMatcher
- **Homography estimation:** Using RANSAC with matched corners
- **3D reconstruction:** Structure-from-Motion with corner tracks